# Assignment Part 3 — Initial Data Population (Seeding)

This notebook demonstrates the `seed_database()` function that populates all tables
with realistic starting data before operations and testing.

In [10]:
import sys
sys.path.insert(0, '../src')

from rich.console import Console
from rich.table import Table
console = Console()

## 1. Run `seed_database()`

In [11]:
from or_scheduler.seed import seed_database

counts = seed_database()
print("\nSeed summary:")
for key, val in counts.items():
    print(f"  {key}: {val}")

Seeding departments...
Seeding rooms...
Seeding equipment...
Seeding staff...
Seeding patients...
Seeding schedules (14 days)...

✅ Seed complete: {'departments': 5, 'rooms': 8, 'equipment': 6, 'staff': 20, 'patients': 100, 'schedules_created': 0}

Seed summary:
  departments: 5
  rooms: 8
  equipment: 6
  staff: 20
  patients: 100
  schedules_created: 0


## 2. Verify Row Counts Per Table

In [12]:
from sqlalchemy import text
from or_scheduler.database import engine

tables_to_count = [
    'departments', 'staff', 'rooms', 'equipment', 'patients',
    'cases', 'appointments', 'room_reservations', 'staff_reservations',
    'equipment_reservations', 'room_schedules', 'staff_schedules',
    'equipment_schedules', 'overrides', 'audit_log'
]

t = Table(title="Row Counts", show_header=True, header_style="bold blue")
t.add_column("Table", style="green")
t.add_column("Rows", justify="right", style="yellow")

with engine.connect() as conn:
    for table_name in tables_to_count:
        row = conn.execute(text(f'SELECT COUNT(*) FROM {table_name}')).scalar()
        t.add_row(table_name, str(row))

console.print(t)

            Row Counts            
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Table                  ┃  Rows ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ departments            │     5 │
│ staff                  │    20 │
│ rooms                  │     8 │
│ equipment              │     6 │
│ patients               │ 10000 │
│ cases                  │ 10555 │
│ appointments           │    89 │
│ room_reservations      │    89 │
│ staff_reservations     │   178 │
│ equipment_reservations │     1 │
│ room_schedules         │   197 │
│ staff_schedules        │   310 │
│ equipment_schedules    │    84 │
│ overrides              │     1 │
│ audit_log              │ 10647 │
└────────────────────────┴───────┘

## 3. Sample Data — Departments

In [13]:
from sqlalchemy.orm import Session
from sqlalchemy import select
from or_scheduler.models import Department

with Session(engine) as session:
    depts = session.execute(select(Department)).scalars().all()

t = Table(title="Departments", show_header=True, header_style="bold cyan")
t.add_column("Name", style="green")
t.add_column("Building")
t.add_column("Floor")
for d in depts:
    t.add_row(d.name, d.building or "", str(d.floor or ""))
console.print(t)

              Departments              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Name           ┃ Building   ┃ Floor ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ ศัลยกรรมกระดูก   │ Building A │ 3     │
│ ศัลยกรรมหัวใจ    │ Building B │ 4     │
│ ประสาทศัลยศาสตร์ │ Building A │ 5     │
│ ศัลยกรรมทั่วไป    │ Building C │ 2     │
│ วิสัญญีวิทยา       │ Building A │ 3     │
└────────────────┴────────────┴───────┘

## 4. Sample Data — Staff by Role

In [14]:
from or_scheduler.models import Staff

with Session(engine) as session:
    staff_list = session.execute(select(Staff).order_by(Staff.role)).scalars().all()

t = Table(title="Staff Members", show_header=True, header_style="bold cyan")
t.add_column("Name", style="green")
t.add_column("Role", style="yellow")
t.add_column("License")
t.add_column("Active")
for s in staff_list:
    t.add_row(s.name, s.role, s.license_number or "N/A", "✓" if s.is_active else "✗")
console.print(t)

                          Staff Members                          
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓
┃ Name                  ┃ Role              ┃ License  ┃ Active ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩
│ นพ.ภาสกร ศุภนิมิตร       │ ANAESTHESIOLOGIST │ MD-01009 │ ✓      │
│ นพ.กิตติ บุญเสริม         │ ANAESTHESIOLOGIST │ MD-01005 │ ✓      │
│ นพ.สุรชาติ แก้วมณี        │ ANAESTHESIOLOGIST │ MD-01006 │ ✓      │
│ นพ.ปิยะ ทองสุข          │ ANAESTHESIOLOGIST │ MD-01007 │ ✓      │
│ นพ.รัชดา วิไลวรรณ       │ ANAESTHESIOLOGIST │ MD-01008 │ ✓      │
│ ประสานงาน ศิริพร ลาภมาก │ COORDINATOR       │ N/A      │ ✓      │
│ ประสานงาน นิภา วิริยะ    │ COORDINATOR       │ N/A      │ ✓      │
│ ประสานงาน วันเพ็ญ ใจงาม │ COORDINATOR       │ N/A      │ ✓      │
│ ประสานงาน สมหญิง พงษ์ศรี │ COORDINATOR       │ N/A      │ ✓      │
│ ประสานงาน มาลี สุดรัก    │ COORDINATOR       │ N/A      │ ✓      │
│ พยาบาล สุนิษา เพชรรัตน์   │ SCRUB_NURSE       │ N/A      │ ✓      │
│ พยาบาล วราภรณ์ ใจดี     │ SCRUB_NURSE       │ N/A      │ ✓      │
│ พยาบาล นันทนา สุขสมาน   │ SCRUB_NURSE       │ N/A      │ ✓      │
│ พยาบาล อัมพร ชัยสิทธิ์     │ SCRUB_NURSE       │ N/A      │ ✓      │
│ พยาบาล ปาริชาต บุญมี     │ SCRUB_NURSE       │ N/A      │ ✓      │
│ นพ.สมชาย วงศ์สุวรรณ     │ SURGEON           │ MD-01000 │ ✓      │
│ นพ.ธีรพงษ์ มณีรัตน์        │ SURGEON           │ MD-01004 │ ✓      │
│ นพ.อนุชา พงษ์พิทักษ์       │ SURGEON           │ MD-01003 │ ✓      │
│ นพ.วิชัย ศรีสมบูรณ์        │ SURGEON           │ MD-01002 │ ✓      │
│ นพ.ประสิทธิ์ รักษ์ไทย      │ SURGEON           │ MD-01001 │ ✓      │
└───────────────────────┴───────────────────┴──────────┴────────┘

## 5. Sample Data — Rooms

In [15]:
from or_scheduler.models import Room

with Session(engine) as session:
    rooms = session.execute(select(Room).order_by(Room.room_code)).scalars().all()

t = Table(title="Operating Rooms", show_header=True, header_style="bold cyan")
t.add_column("Code", style="green")
t.add_column("Type", style="yellow")
t.add_column("Laminar Flow")
t.add_column("Active")
for r in rooms:
    t.add_row(r.room_code, r.room_type, "✓" if r.is_laminar_flow else "", "✓" if r.is_active else "✗")
console.print(t)

                Operating Rooms                 
┏━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Code     ┃ Type      ┃ Laminar Flow ┃ Active ┃
┡━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━┩
│ ER-1     │ EMERGENCY │              │ ✓      │
│ HYBRID-1 │ HYBRID    │ ✓            │ ✓      │
│ OR-1     │ OR        │ ✓            │ ✓      │
│ OR-2     │ OR        │ ✓            │ ✓      │
│ OR-3     │ OR        │ ✓            │ ✓      │
│ OR-4     │ OR        │ ✓            │ ✓      │
│ OR-5     │ OR        │ ✓            │ ✓      │
│ OR-6     │ OR        │ ✓            │ ✓      │
└──────────┴───────────┴──────────────┴────────┘

## 6. Sample Data — Equipment

In [16]:
from or_scheduler.models import Equipment

with Session(engine) as session:
    equips = session.execute(select(Equipment)).scalars().all()

t = Table(title="Equipment Units", show_header=True, header_style="bold cyan")
t.add_column("Serial", style="green")
t.add_column("Type", style="yellow")
t.add_column("Status")
t.add_column("Sterilization (min)")
for e in equips:
    t.add_row(e.serial_number, e.equipment_type, e.status, str(e.sterilization_duration_min))
console.print(t)

                                    Equipment Units                                     
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Serial        ┃ Type                             ┃ Status      ┃ Sterilization (min) ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ CARM-002      │ C-arm Fluoroscopy                │ AVAILABLE   │ 30                  │
│ LAPC-001      │ Laparoscopic Tower               │ AVAILABLE   │ 45                  │
│ LAPC-002      │ Laparoscopic Tower               │ AVAILABLE   │ 45                  │
│ DAVINCI-001   │ Robotic Surgical System da Vinci │ AVAILABLE   │ 60                  │
│ CELLSAVER-001 │ Cell Saver                       │ AVAILABLE   │ 20                  │
│ CARM-001      │ C-arm Fluoroscopy                │ STERILIZING │ 30                  │
└───────────────┴──────────────────────────────────┴─────────────┴─────────────────────┘

## 7. Sample Patients (first 10)

In [17]:
from or_scheduler.models import Patient

with Session(engine) as session:
    patients = session.execute(select(Patient).limit(10)).scalars().all()

t = Table(title="Patients (sample)", show_header=True, header_style="bold cyan")
t.add_column("HN", style="green")
t.add_column("Name")
t.add_column("Age")
t.add_column("Blood Type")
t.add_column("Allergies")
for p in patients:
    t.add_row(p.hn, p.name, str(p.age or ""), p.blood_type or "", p.allergies or "None")
console.print(t)
print(f"\n✅ 100 patients total in database.")

                        Patients (sample)                        
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ HN          ┃ Name             ┃ Age ┃ Blood Type ┃ Allergies ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ HN-00000001 │ Caitlin Woodward │ 32  │ A+         │ Latex     │
│ HN-00000002 │ Frank Cochran    │ 53  │ B-         │ None      │
│ HN-00000003 │ Leslie Kelley    │ 35  │ A-         │ Latex     │
│ HN-00000004 │ Robin Spencer    │ 29  │ AB+        │ None      │
│ HN-00000005 │ Jasmin Rogers    │ 21  │ A-         │ None      │
│ HN-00000006 │ Kaylee Wallace   │ 47  │ A+         │ Aspirin   │
│ HN-00000007 │ Douglas Gibbs    │ 43  │ AB+        │ None      │
│ HN-00000008 │ James Perkins    │ 75  │ O+         │ None      │
│ HN-00000009 │ John Villanueva  │ 38  │ AB+        │ None      │
│ HN-00000010 │ Lorraine Ellis   │ 53  │ B+         │ None      │
└─────────────┴──────────────────┴─────┴────────────┴───────────┘


✅ 100 patients total in database.


## 8. Idempotency Check — Run Seed Again

In [18]:
# Running seed_database() again should return the same counts — no duplicates
counts2 = seed_database()
print("Second run (idempotency check):")
for key, val in counts2.items():
    print(f"  {key}: {val}")

# Verify the 100 SEED patients (HN-XXXXXXXX) were not duplicated.
# Note: NB04 may add PERF-XXXXXXXX patients for performance testing — those are
# expected and do not affect seed idempotency.
with engine.connect() as conn:
    seed_patient_count = conn.execute(
        text("SELECT COUNT(*) FROM patients WHERE hn LIKE 'HN-%'")
    ).scalar()
print(f"\nSeed patient count (HN-*) after second seed: {seed_patient_count} (should still be 100)")
assert seed_patient_count == 100, "Idempotency broken — duplicate HN seed patients detected!"
print("✅ Idempotency confirmed — no duplicates created.")

Seeding departments...
Seeding rooms...
Seeding equipment...
Seeding staff...
Seeding patients...
Seeding schedules (14 days)...

✅ Seed complete: {'departments': 5, 'rooms': 8, 'equipment': 6, 'staff': 20, 'patients': 100, 'schedules_created': 0}
Second run (idempotency check):
  departments: 5
  rooms: 8
  equipment: 6
  staff: 20
  patients: 100
  schedules_created: 0

Seed patient count (HN-*) after second seed: 100 (should still be 100)
✅ Idempotency confirmed — no duplicates created.
